# Probe Guidance: Descriptive Statistics for the LL Probe Guidance Dataset

In [1]:
import sys
sys.path.insert(0, "/home/jack/code/vjepa2-probe-guidance/vjepa2")
print(sys.path)

['/home/jack/code/vjepa2-probe-guidance/vjepa2', '/usr/lib/python310.zip', '/usr/lib/python3.10', '/usr/lib/python3.10/lib-dynload', '', '/home/jack/code/vjepa2-probe-guidance/.venv/lib/python3.10/site-packages', '/home/jack/code/vjepa2-probe-guidance/vjepa2/src', '/home/jack/code/vjepa2-probe-guidance/.venv/lib/python3.10/site-packages/rerun_sdk']


In [2]:
import numpy as np
import torch
import torchvision.transforms.functional as F
from tqdm import tqdm

from app.vjepa_ll_probe_guidance.ll_probe_guidance import LLProbeGuidanceDataset
from app.vjepa_ll_probe_guidance.transforms import make_transforms

In [3]:
def load_clips(sample, device):
    clips = sample[0].to(device, non_blocking=True)  # [B C T H W]
    actions = sample[1].to(device, non_blocking=True)  # [B T-1 6]
    states = sample[2].to(device, non_blocking=True)  # [B T 6]
    extrinsics = sample[3].to(device, non_blocking=True)  # [B T 6]
    return (clips, actions, states, extrinsics)

In [4]:
crop_size = 256
tokens_per_frame = 256
clip_size = 8
fps = 4

transform = make_transforms(
    crop_size=crop_size,
)

train_dataset = LLProbeGuidanceDataset(
    data_root="/home/jack/data/probe_guidance_dataset_june/train",
    frames_per_clip=clip_size,
    frame_skip=1,
    frames_per_second=fps,
    transform=transform,
    is_train=False,
)

loader = torch.utils.data.DataLoader(
    train_dataset,
    shuffle=True,
    batch_size=1,
    num_workers=32,
)

Scanning 60 episodes for valid tracking clips...


100%|████████████████████████████████████████████████████████████████████████████████████████████████████| 60/60 [00:01<00:00, 51.87it/s]

Retained 60 episodes.


In [7]:
sample = next(iter(loader))
clips, _, _, _ = load_clips(sample, "cpu")

mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1, 1)
clips = clips.squeeze()

denormed_clips = (clips * std) + mean
denormed_clips.clamp(0.0, 1.0)

frames = denormed_clips.unbind(dim=1)

In [8]:
for i, frame in enumerate(frames):
    img = F.to_pil_image(frame.squeeze())
    img.save(f"frame_{i}.png")